**Github repository:** https://github.com/NicholasBorch/ComSocSci-Assignments

**Group Members:** Nicholas Borch (s234841) and Robin Braagaard (s234856)

**Distribution of Contribution:** All group members contributed equally in the planning and execution of solutions and implementation of code for the assignment tasks.

------

# 02467 Computational social science - Assignment 1

------

### Python libraries used in assigment

In [ ]:
### Importing libraries
from pathlib import Path
import pandas as pd
import re
import unicodedata
import time
from typing import Dict, Any, List, Optional, Tuple, Set
import numpy as np
import requests
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from unidecode import unidecode
from rapidfuzz import fuzz
from joblib import Parallel, delayed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from contextlib import contextmanager
import json

----

## Part 1: Ready Made vs Custom Made Data

> **Exercise: Ready made data vs Custom made data** In this exercise, I want to make sure you have understood they key points of my lecture and the reading. 

> 1. What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)? You can support your arguments based on the content of the lecture and the information you read in Chapter 2.3 of the book __(answer in max 150 words)__.

#### Answer for P1Q1:

**Centola (Custom-made data):**

**Pros:** Strong causal clarity by having the network differences as the only causal variable. Experiment is designed explicitly around the research question. Participants self-opted into the platform following advertisements on health websites, not much suggests experiment awareness, suggesting a degree of non-reactivity.

**Cons:** Small scale (1,528 participants). Signing up to a forum is “low-cost”, genuine engagement is not captured; some may have signed up merely due to email annoyance. Study is unrepresentative, limited to internet users. Difficult to replicate as the platform is inaccessible to others.

**Nicolaides (Ready-made data):**

**Pros:** Massive scale (1.1M users, 350M+ km). Always-on continuous tracking over 5 years enables observation of emerging patterns. Based on real behaviour, not self-reports.

**Cons:** Non-representative, focusing on fitness-app users, probably an atypically health-motivated and competitive demographic, possibly exercising to beat friends rather than due to contagion. Algorithmic confounding likely, as the platform may artificially promote friend activity.



> 2. How do you think these differences can influence the interpretation of the results in each study? __(answer in max 150 words)__

#### Answer for P1Q2:

**Centola:** The small scale and artificial setting limit the generalisability of the experiment, results may not hold in larger, real-world networks. The low-cost behaviour that is being analyzed means we cannot conclude that clustered networks would similarly spread larger behavioural changes. However, the experimental control means the conclusions about network activity presented are highly credible within the study's scope.

**Nicolaides:** The massive scale and real-world setting strengthen generalisation. However, the non-representative sample of fitness-app users means results may not generalise too well to the rest of the population, which casts a few doubts on what the study finds the impact of contagion to be. Algorithmic confounding introduces uncertainty about whether observed contagion reflects genuine social influence or is being artificially pushed and constructed by the platform motivation. The reliance on behavioural data without further context means we cannot completely decide whether motivation or competitive pressure drives the observed behaviour.

------

## Pre Part 2 (Useful info and code from week 1):

Collecting the author names from the International Conference in Computational Social Science 2025.

Website URL: https://ic2s2-2025.org/program/

In [ ]:
# Output folder for Assignment 1 datasets
A1_DATA_DIR = Path("A1_data")
A1_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Source CSV for Web-scraping the list of participants to the International Conference in Computational Social Science (found via Network tab in browser dev tools)
IC2S2_SCHEDULE_CSV_URL = "https://ic2s2-2025.org/files/ic2s2_2025_schedule_v5.csv"

In [ ]:
# Setting up a function to canonicalize names, so that any spelling errors or differences in formatting 
# can be standardized for easier comparison and analysis and to avoid duplicates of the same person 
# being treated as different individuals.
def canon(name: str) -> str:
    """
    Canonical form of a person name for de-duplication:
    - lower-case
    - remove accents
    - transliterate remaining non-ASCII (ß->ss, ø->o, curly quotes->')
    - remove punctuation
    - collapse whitespace
    """
    if name is None:
        return ""
    name = str(name).strip().lower()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = unidecode(name)
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name.strip()

In [ ]:
### Loading IC2S2 schedule CSV
df_schedule = pd.read_csv(IC2S2_SCHEDULE_CSV_URL, storage_options={"User-Agent": "Mozilla/5.0"})
print("Rows:", len(df_schedule))
df_schedule.head()

In [ ]:
### Extracting authors and chairs into one raw name series
authors = df_schedule["author"].dropna().astype(str).str.split(",").explode().str.strip()
authors = authors[authors.ne("")]

chairs = df_schedule["chair"].dropna().astype(str).str.split(",").explode().str.strip()
chairs = chairs[chairs.ne("")]

# Combining the two lists
names_raw = pd.concat([authors, chairs], ignore_index=True)

print("Raw author entries:", len(authors))
print("Raw chair entries :", len(chairs))
print("Raw combined entries:", len(names_raw))
print("Unique (raw, exact string):", names_raw.nunique())
names_raw.head(10)

In [ ]:
# Creating a DataFrame that maps each raw name collected from the website 
# to its canonical version in order to standardize names and find 
# duplicates that may appear with slightly different spellings (e.g., accents, 
# punctuation differences, extra spaces). 
# Thereafter, counting how many unique canonical names exist (true unique researchers).
# and identify canonical names that correspond to multiple raw spellings.
names_df = pd.DataFrame({"name_raw": names_raw})
names_df["name_canon"] = names_df["name_raw"].map(canon)
print("Unique canonical names:", names_df["name_canon"].nunique())

# Canonical names that map to multiple raw spellings
variants = (names_df.groupby("name_canon")["name_raw"].unique().reset_index())
variants["n_variants"] = variants["name_raw"].apply(len)
variants_multi = variants[variants["n_variants"] > 1].sort_values("n_variants", ascending=False)

print("Canonical names with >1 raw variant:", len(variants_multi))
variants_multi

In [ ]:
# Setting up the final list of unique IC2S2 researchers.
# Due to multiple raw spellings that map to the same canonical name,
# raw names are sorted alphabetically and only one representative
# raw name per canonical version is kept.
unique_researchers = (names_df.sort_values("name_raw").drop_duplicates("name_canon")["name_raw"].reset_index(drop=True))

out_unique = A1_DATA_DIR / "D0_unique_researchers.csv"
unique_researchers.to_csv(out_unique, index=False, header=["name"])

print("Unique researchers:", len(unique_researchers))
unique_researchers.head(20)

In [ ]:
# In Week 2 (Part 2) text embeddings are used, hence a useful "text profile" is prepared for each person.

# For every row in the IC2S2 schedule (a talk/poster/etc.), a short document consisting of "title. abstract" is created.
# That document is then assigned to every author and chair listed on that row.

# This results in one text blob per researcher that summarizes what they contributed to at IC2S2 2025,
# and is used to find correct author ID in Part 2 of the assignment.
df_temp = df_schedule[["title", "abstract", "author", "chair"]].copy()
for col in ["title", "abstract", "author", "chair"]:
    df_temp[col] = df_temp[col].fillna("").astype(str)

rows = []
for _, r in df_temp.iterrows():
    paper_text = (r["title"].strip() + ". " + r["abstract"].strip()).strip()
    people = []
    people += [x.strip() for x in r["author"].split(",") if x.strip()]
    people += [x.strip() for x in r["chair"].split(",") if x.strip()]
    for person in people:
        rows.append({"name_raw": person, "name_canon": canon(person), "paper_text": paper_text})

df_people_text_long = pd.DataFrame(rows)

# Collecting all paper_text per person
df_people_text = (df_people_text_long.groupby("name_canon")["paper_text"].apply(lambda s: " ".join([t for t in s.tolist() if t])).reset_index().rename(columns={"paper_text": "conference_person_text"}))

out_person_text = A1_DATA_DIR / "conference_person_text.csv"
df_people_text.to_csv(out_person_text, index=False)

print("People with text:", len(df_people_text))
df_people_text.head()

## Part 2: Find Researchers using the OpenAlex API

> **Exercise 3: Find potential Computational Social Scientists** In this exercise, we'll use the OpenAlex API to compile a list of researchers in the field of Computational Social Science, focusing on those who have attended the IC2S2 conference in 2025.

> 1. **Retreive data.** Consider the set of unique researcher names that you collected in Week 1, Exercise 3. Use the _authors_ endpoint of the [OpenAlex API](https://docs.openalex.org/api-entities/authors) to _search_ these researchers in the database based on their names. Loop through the list and, for each researcher in your list, find: 
>     - their _id_: The OpenAlex ID for this author.
>     - their _display\_name_: The name of the author as a single string.
>     - their _works\_api\_url_: A URL that will get you a list of all this author's works.
>     - their _h\_index_ : The h-index for this author.
>     - their _works\_count_: The number of  Works this this author has created.
>     - their _country\_code_: The country code of their last known institution

#### Answer for P2Q1:



OpenAlex API was used to collect author information.

OpenAlex API URL: https://docs.openalex.org/how-to-use-the-api/api-overview

In [ ]:
# OpenAlex API
API_KEY = "jGoPhEc89mvkhWcjqi6PbD"
HEADERS = {"User-Agent": "Nicholas B (mailto:nicholas.borch@gmail.com)"}

OPENALEX_AUTHORS_URL = "https://api.openalex.org/authors"
OPENALEX_WORKS_URL = "https://api.openalex.org/works"

# Embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
### Loading D0 names and prepared conference_person_text
df_unique = pd.read_csv(A1_DATA_DIR / "D0_unique_researchers.csv")
df_person = pd.read_csv(A1_DATA_DIR / "conference_person_text.csv")

person_text_lookup = dict(zip(df_person["name_canon"].astype(str), df_person["conference_person_text"].astype(str)))
df_unique.head()

In [ ]:
# Name similarity scoring:

# OpenAlex often returns many candidate authors for a queried name,
# including both plausible matches and clearly unrelated individuals.
# We opted to first apply a fuzzy name similarity filter to ensure that only sufficiently
# similar names are considered. After this pre-filtering step, semantic embeddings of 
# research content is utilized to identify the most likely correct author.

def name_score(query: str, candidate: str) -> float:
    q = canon(query)
    c = canon(candidate)

    full = fuzz.token_set_ratio(q, c) / 100.0

    q_first = q.split()[0] if q.split() else ""
    c_first = c.split()[0] if c.split() else ""
    first = (fuzz.ratio(q_first, c_first) / 100.0) if (q_first and c_first) else 0.0

    return 0.7 * full + 0.3 * first

def safe_get(url: str, params: Optional[dict] = None, headers: Optional[dict] = None,
             retries: int = 5, base_wait: float = 1.5, timeout: int = 20) -> Dict[str, Any]:
    params = params or {}
    headers = headers or {}

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)

            # backoff on rate limit
            if r.status_code == 429:
                time.sleep(base_wait * (2 ** (attempt - 1)))
                continue

            r.raise_for_status()
            return r.json()

        except Exception:
            time.sleep(base_wait * (2 ** (attempt - 1)))

    return {}

# Caches for speed and fewer API calls
author_search_cache: Dict[str, List[dict]] = {}
works_titles_cache: Dict[str, List[str]] = {}
conf_vec_cache: Dict[str, np.ndarray] = {}

In [ ]:
# This cell extracts exactly the fields required by the assignment:
#   - id
#   - display_name
#   - works_api_url
#   - h_index
#   - works_count
#   - country_code (from last_known_institution)

def search_authors(name: str, per_page: int = 30) -> List[dict]:
    if name in author_search_cache:
        return author_search_cache[name]

    data = safe_get(OPENALEX_AUTHORS_URL, params={"search": name, "per-page": per_page, "api_key": API_KEY}, headers=HEADERS)
    results = data.get("results", []) or []
    author_search_cache[name] = results
    return results

def author_id_short(author_obj: dict) -> str:
    aid = author_obj.get("id", "")
    return str(aid).rstrip("/").split("/")[-1] if aid else ""

def get_country_code(author_obj: dict) -> Optional[str]:
    # Country code of last known institution
    lki = author_obj.get("last_known_institution")
    if isinstance(lki, dict) and lki.get("country_code"):
        return lki.get("country_code")
    return None

def output_row_for_assignment(searched_name: str, author_obj: Optional[dict]) -> Dict[str, Any]:
    # Columns requested in Part 2
    if not isinstance(author_obj, dict):
        return {
            "searched_name": searched_name,
            "id": None,
            "display_name": None,
            "works_api_url": None,
            "h_index": None,
            "works_count": None,
            "country_code": None,
        }

    return {
        "searched_name": searched_name,
        "id": author_obj.get("id"),
        "display_name": author_obj.get("display_name"),
        "works_api_url": author_obj.get("works_api_url"),
        "h_index": (author_obj.get("summary_stats") or {}).get("h_index"),
        "works_count": author_obj.get("works_count"),
        "country_code": get_country_code(author_obj),
    }

In [ ]:
# For the embedding model the last 15 works titles is fetched for author context.
# Cursor pagination is utilized for this.
def fetch_last_15_titles(author_obj: dict, max_titles: int = 15) -> List[str]:
    aid = author_id_short(author_obj)
    if not aid:
        return []

    if aid in works_titles_cache:
        return works_titles_cache[aid]

    titles: List[str] = []
    cursor = "*"
    per_page = 200

    while cursor and len(titles) < max_titles:
        data = safe_get(
            OPENALEX_WORKS_URL,
            params={
                "filter": f"authorships.author.id:{aid}",
                "sort": "publication_date:desc",
                "per-page": per_page,
                "cursor": cursor,
                "select": "title,publication_date",
                "api_key": API_KEY,
            },
            headers=HEADERS
        )

        results = data.get("results", []) or []
        if not results:
            break

        for w in results:
            t = w.get("title")
            if t:
                titles.append(t)
                if len(titles) >= max_titles:
                    break

        cursor = data.get("meta", {}).get("next_cursor")

    titles = titles[:max_titles]
    works_titles_cache[aid] = titles
    return titles

In [ ]:
# Candidate embedding construction:
# For each OpenAlex author candidate, a semantic representation is built
# by concatenating (1) high-level topics, (2) more granular concepts, and
# (3) titles of the 15 most recent publications.

# If some components are missing, whatever information exists is embedded. 
# If all components are missing, semantic embedding is skipped for that candidate
# and instead we rely on the highest name similarity score.

def candidate_embedding_text(author_obj: dict) -> str:
    topics = " ".join([t.get("display_name", "") for t in (author_obj.get("topics", []) or []) if isinstance(t, dict)]).strip()

    concepts = " ".join([c.get("display_name", "") for c in (author_obj.get("x_concepts", []) or []) if isinstance(c, dict)]).strip()

    titles = fetch_last_15_titles(author_obj, max_titles=15)
    recent_works = " ".join([t for t in titles if isinstance(t, str) and t.strip()]).strip()

    return " ".join([topics, concepts, recent_works]).strip()

In [ ]:
# Match one IC2S2 conference author

# For each conference name:
# 1) Qery OpenAlex to retrieve candidate authors.
# 2) Pre-filter candidates by fuzzy name similarity (name_score > 0.4) to remove clearly unrelated individuals. 
#    Threshold is chosen to still include names consisting of initials or abreviations (based on a small random sampling inspection in weekly exercises).
# 3) If the conference_person_text exists and at least one candidate has embeddable text (topics, concepts, recent works):
#    - Compute embeddings for both the conference text and each candidate.
#    - And select the candidate with the highest cosine similarity to the conference text.
#    Otherwise:
#       - Fall back to selecting the candidate with the highest name similarity score.

def get_conf_vec(name_canon: str, conf_text: str) -> Optional[np.ndarray]:
    if not conf_text or not str(conf_text).strip():
        return None
    if name_canon in conf_vec_cache:
        return conf_vec_cache[name_canon]
    v = model.encode(conf_text)
    conf_vec_cache[name_canon] = v
    return v

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def match_one_author(name_raw: str, conf_text: str) -> Optional[dict]:
    candidates = search_authors(name_raw)
    if not candidates:
        return None

    filtered: List[Tuple[float, dict]] = []
    for a in candidates:
        display = a.get("display_name", "")
        s = name_score(name_raw, display)
        if s > 0.4:
            filtered.append((s, a))

    if not filtered:
        return None

    nc = canon(name_raw)
    conf_vec = get_conf_vec(nc, conf_text)

    # No conference context, then fallback to name score
    if conf_vec is None:
        return max(filtered, key=lambda x: x[0])[1]

    # Embeddable candidate texts setup
    emb_texts: List[str] = []
    emb_authors: List[dict] = []
    for _, a in filtered:
        txt = candidate_embedding_text(a)
        if txt.strip():
            emb_texts.append(txt)
            emb_authors.append(a)

    # If no candidate has any embeddable text, then fallback to name score
    if len(emb_authors) == 0:
        return max(filtered, key=lambda x: x[0])[1]

    cand_vecs = model.encode(emb_texts, batch_size=16)
    sims = [cosine(conf_vec, v) for v in cand_vecs]

    best_idx = int(np.argmax(sims))
    return emb_authors[best_idx]

> 2. **Data Storage** Store this information in a Pandas DataFrame and save it to file.

#### Answer for P2Q2:

In [ ]:
# Creating the DataFrame with the required columns and save it as a D1 dataset in A1_data/.
# NOTE: Takes a long time to run.

rows: List[Dict[str, Any]] = []

for name_raw in tqdm(df_unique["name"].astype(str).tolist(), desc="Part 2: OpenAlex author matching"):
    conf_text = person_text_lookup.get(canon(name_raw), "")
    best = match_one_author(name_raw=name_raw, conf_text=conf_text)
    rows.append(output_row_for_assignment(searched_name=name_raw, author_obj=best))

df_D1 = pd.DataFrame(rows)
df_D1.head()

In [ ]:
# Saving D1 dataset to file
D1_PATH = A1_DATA_DIR / "D1_openalex_authors.csv"
df_D1.to_csv(D1_PATH, index=False)
print("Saved:", D1_PATH)